# Suite ADJ — Grid adjacency

An adjacency spec from one child to another moves the other beside it, along its rotation quantized to a quarter turn, where one change of the grid's sequence can, or into the cell beside it that a spec along the same side placed, which they then share; removing the spec reverts the child to its original insertion position: mbse-basis [CAP-239: grid adjacency](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-239), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import { Builders, Validators } from "@mbse/schemas/Framework";

import { Container, Logical } from "../src/Framework/index.js";

function assert(condition: unknown, message = "assertion failed"): asserts condition {
  if (!condition) throw new Error(`AssertionError: ${message}`);
}

const store: any = new Builders.Store();
store.transaction(Logical.install);
store.transaction(Container.install);
const L = store.Logical;
const diagrams = store.singleton("Logical.Diagrams");
const problems = (schema: any, root: any): string[] => Validators.Validate(store).Reachable(schema, root);
const text = (value: unknown): string => JSON.stringify(value, (_k, v) => typeof v === "bigint" ? Number(v) : v);

store.transaction(() =>
  L.DiagramDirectory(diagrams, (f: any) => f.diagrams((a: any) => a.insert((e: any) => e.name("overview").diagram((d: any) => d)))));
const overview = diagrams.diagrams({ name: "overview" });

/** A top-level shape of the diagram holding a child of each name in `children`, laid out in a grid. */
function box(name: string, children: string, span: bigint, grows = "rows", cell_rotation = 0): any {
  const spec = (s: any): any => {
    for (const child of children) s.children((k: any) => k.insert((e: any) => e.name(child).child((c: any) => c)));
    return s;
  };
  let shape: any;
  store.transaction(() => {
    L.OfDiagram(overview, (d: any) => d.shapes((s: any) => s.insert((e: any) => e.name(name).shape(spec))));
    shape = overview.shapes({ name });
    Container.layout(shape, span, grows, cell_rotation);
  });
  return shape;
}

/** Adds the adjacency spec from child `a` to child `b`, by `degrees`, and adjoins them: whether it did. */
function beside(shape: any, a: string, b: string, degrees: number): boolean {
  let done = false;
  store.transaction(() => {
    L.OfShape(shape.children({ name: a }), (s: any) => s.adjacent_to((x: any) => x.insert(
      (e: any) => e.to_shape(shape.children({ name: b })).rotation((r: any) => r.degrees(degrees)))));
    done = Container.adjoined(shape.children({ name: a }), shape.children({ name: b }));
  });
  return done;
}

/** Removes the adjacency spec from child `a` to child `b`, and unjoins `b`. */
function apart(shape: any, a: string, b: string): void {
  store.transaction(() => {
    L.OfShape(shape.children({ name: a }), (s: any) => s.adjacent_to((x: any) => x.remove(
      (e: any) => e.to_shape(shape.children({ name: b })))));
    Container.unjoined(shape.children({ name: b }));
  });
}

/** Each cell, in row and column order, as JSON text: its row and column, the names of the children it holds, in
 * order, each with its order, and its axis and extents. */
function cells(shape: any): string {
  const [entry] = [...shape.grid];
  const names = new Map([...shape.children].map((e: any) => [e.child, e.name.get()]));
  const found = [...entry.grid.cells]
    .sort((x: any, y: any) => Number(x.row.get() - y.row.get()) || Number(x.column.get() - y.column.get()))
    .map((e: any) => [[e.row.get(), e.column.get()], [...e.cell.children].map((h: any) => [names.get(h.shape), h.order.get()]),
      e.cell.grows.get(), e.cell.rows.get(), e.cell.columns.get()]);
  return text(found);
}

<a id="ADJ-01"></a>
## ADJ-01 · A rotation is quantized to the nearest quarter turn, half way to the clockwise one

In [ ]:
assert(text([0, 44.9, 45, 134.9, 190, 270, 315, -90, 359.5, 720].map((d) => Container.quarter(d))) === "[\"north\",\"north\",\"east\",\"east\",\"south\",\"west\",\"north\",\"west\",\"north\",\"north\"]");

<a id="ADJ-02"></a>
## ADJ-02 · A spec moves its child beside the other: east, then south, by one change of the sequence each

In [ ]:
const cpu = box("cpu", "abcde", 3n);
assert(cells(cpu) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"b\",1]],null,1,1],[[0,2],[[\"c\",2]],null,1,1],[[1,0],[[\"d\",3]],null,1,1],[[1,1],[[\"e\",4]],null,1,1]]");
assert(text(beside(cpu, "a", "e", 90)) === "true");
assert(cells(cpu) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"e\",4]],null,1,1],[[0,2],[[\"b\",1]],null,1,1],[[1,0],[[\"c\",2]],null,1,1],[[1,1],[[\"d\",3]],null,1,1]]");
assert(text(beside(cpu, "a", "d", 190)) === "true");
assert(cells(cpu) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"e\",4]],null,1,1],[[0,2],[[\"b\",1]],null,1,1],[[1,0],[[\"d\",3]],null,1,1],[[1,1],[[\"c\",2]],null,1,1]]");

<a id="ADJ-03"></a>
## ADJ-03 · Two specs from one child along one side share a cell, which grows along its grid's axis; a spec it cannot meet stays

In [ ]:
assert(text(beside(cpu, "a", "c", 80)) === "true");
assert(cells(cpu) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"e\",4],[\"c\",2]],\"rows\",2,1],[[0,2],[[\"b\",1]],null,1,1],[[1,0],[[\"d\",3]],null,1,1]]");
const io = box("io", "pqrst", 3n);
assert(text([beside(io, "p", "r", 180), beside(io, "p", "t", 180), beside(io, "s", "r", 90)]) === "[true,true,false]");
assert(cells(io) === "[[[0,0],[[\"p\",0]],null,1,1],[[0,1],[[\"q\",1]],null,1,1],[[0,2],[[\"s\",3]],null,1,1],[[1,0],[[\"r\",2],[\"t\",4]],\"rows\",2,1]]");

<a id="ADJ-04"></a>
## ADJ-04 · Removing a spec reverts its child to its original insertion position, then applies the specs that remain

In [ ]:
apart(cpu, "a", "e");
assert(cells(cpu) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"c\",2]],null,1,1],[[0,2],[[\"b\",1]],null,1,1],[[1,0],[[\"d\",3]],null,1,1],[[1,1],[[\"e\",4]],null,1,1]]");
apart(cpu, "a", "c");
assert(cells(cpu) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"b\",1]],null,1,1],[[0,2],[[\"c\",2]],null,1,1],[[1,0],[[\"d\",3]],null,1,1],[[1,1],[[\"e\",4]],null,1,1]]");
apart(io, "p", "r");
assert(cells(io) === "[[[0,0],[[\"p\",0]],null,1,1],[[0,1],[[\"q\",1]],null,1,1],[[0,2],[[\"r\",2]],null,1,1],[[1,0],[[\"s\",3]],null,1,1],[[1,1],[[\"t\",4]],null,1,1]]");

<a id="ADJ-05"></a>
## ADJ-05 · A spec the grid cannot meet is ignored

In [ ]:
const bus = box("bus", "abcde", 3n);
assert(text([beside(bus, "a", "e", 0), beside(bus, "c", "e", 90), beside(bus, "c", "e", 180)]) === "[false,false,false]");
assert(cells(bus) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"b\",1]],null,1,1],[[0,2],[[\"c\",2]],null,1,1],[[1,0],[[\"d\",3]],null,1,1],[[1,1],[[\"e\",4]],null,1,1]]");

<a id="ADJ-06"></a>
## ADJ-06 · A grid that grows columns: east is a column on, north and south within a column

In [ ]:
const col = box("col", "abcd", 2n, "columns");
assert(cells(col) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"c\",2]],null,1,1],[[1,0],[[\"b\",1]],null,1,1],[[1,1],[[\"d\",3]],null,1,1]]");
assert(text(beside(col, "a", "d", 90)) === "true");
assert(cells(col) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"d\",3]],null,1,1],[[1,0],[[\"b\",1]],null,1,1],[[1,1],[[\"c\",2]],null,1,1]]");
assert(text(beside(col, "d", "b", 0)) === "false");
assert(cells(col) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"d\",3]],null,1,1],[[1,0],[[\"b\",1]],null,1,1],[[1,1],[[\"c\",2]],null,1,1]]");

<a id="ADJ-07"></a>
## ADJ-07 · Original insertion positions outlive removals and additions; a grid placed on a container applies its specs

In [ ]:
const alu = box("alu", "xyz", 3n);
assert(text(beside(alu, "y", "z", 270)) === "true");
assert(cells(alu) === "[[[0,0],[[\"x\",0]],null,1,1],[[0,1],[[\"z\",2]],null,1,1],[[0,2],[[\"y\",1]],null,1,1]]");
store.transaction(() => {
  L.OfShape(alu, (s: any) => s.children((k: any) => k.remove((e: any) => e.name("x"))));
  Container.removed(alu);
});
assert(cells(alu) === "[[[0,0],[[\"z\",2]],null,1,1],[[0,1],[[\"y\",1]],null,1,1]]");
store.transaction(() => {
  L.OfShape(alu, (s: any) => s.children((k: any) => k.insert((e: any) => e.name("w").child((c: any) => c))));
  Container.added(alu.children({ name: "w" }));
});
assert(cells(alu) === "[[[0,0],[[\"z\",2]],null,1,1],[[0,1],[[\"y\",1]],null,1,1],[[0,2],[[\"w\",3]],null,1,1]]");
apart(alu, "y", "z");
assert(cells(alu) === "[[[0,0],[[\"y\",1]],null,1,1],[[0,1],[[\"z\",2]],null,1,1],[[0,2],[[\"w\",3]],null,1,1]]");
let late: any;
store.transaction(() => {
  L.OfDiagram(overview, (d: any) => d.shapes((s: any) => s.insert((e: any) => e.name("late").shape((x: any) => x.children(
    (k: any) => k.insert((e: any) => e.name("p").child((c: any) => c)).insert((e: any) => e.name("q").child((c: any) => c))
      .insert((e: any) => e.name("r").child((c: any) => c)))))));
  late = overview.shapes({ name: "late" });
  L.OfShape(late.children({ name: "p" }), (s: any) => s.adjacent_to((x: any) => x.insert(
    (e: any) => e.to_shape(late.children({ name: "r" })).rotation((r: any) => r.degrees(180)))));
  Container.layout(late, 2n);
});
assert(cells(late) === "[[[0,0],[[\"p\",0]],null,1,1],[[0,1],[[\"q\",1]],null,1,1],[[1,0],[[\"r\",2]],null,1,1]]");
assert(text([cpu, io, bus, col, alu, late].map((s) => problems(L.OfShape, s))) === "[[],[],[],[],[],[]]");

<a id="ADJ-08"></a>
## ADJ-08 · A grid's cells are turned by its cell rotation, and with them the compasses of the shapes they hold

In [ ]:
const rot = box("rot", "ab", 2n, "rows", 90);
store.transaction(() => {
  for (const name of "ab") {
    L.OfShape(rot.children({ name }), (s: any) => s.origin((a: any) => a.insert((e: any) => e.point((p: any) => p)))
      .borders((a: any) => a.insert((e: any) => e.name("n").point((p: any) => p)))
      .children((k: any) => k.insert((e: any) => e.name("inner").child((c: any) => c.origin(
        (a: any) => a.insert((e: any) => e.point((p: any) => p)))))));
    const child = rot.children({ name });
    L.OfPoint([...child.origin][0].point, (p: any) => p.outgoing((a: any) => a.insert(
      (e: any) => e.end(child.borders({ name: "n" })).rotation((r: any) => r.degrees(90)))));
    Container.layout(child, 1n, "columns", 45);
  }
});
const a = rot.children({ name: "a" });
const inner = a.children({ name: "inner" });
assert(text([Logical.compass(a), Logical.compass(a.borders({ name: "n" })), Logical.compass(inner)]) === "[90,180,135]");
assert(text(beside(rot, "a", "b", 180)) === "false");
assert(cells(rot) === "[[[0,0],[[\"a\",0]],null,1,1],[[0,1],[[\"b\",1]],null,1,1]]");
store.transaction(() => Container.rotate(rot, 270));
assert(text([Logical.compass(a), Logical.compass(a.borders({ name: "n" })), Logical.compass(inner)]) === "[270,0,315]");